In [1]:
!pip install -q langchain-groq
%pip install -q langchain-classic

In [10]:
from langchain_groq import ChatGroq
from pathlib import Path
import os
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser
from langchain_classic.output_parsers import DatetimeOutputParser

In [ ]:
def _read_env_file(name, filename="secrets.env"):
    """Look for KEY=value in secrets.env in the working dir or /content."""
    for folder in (Path.cwd(), Path("/content")):
        path = folder / filename
        if path.is_file():
            for line in path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, _, val = line.partition("=")
                if key.strip().removeprefix("export ").strip() == name:
                    return val.strip().strip('"').strip("'")
    return None

def get_secret(name, required=True):
    value = None
    try:
        value = userdata.get(name)      # works in the Colab web UI
    except Exception:
        pass                            # timeout / not found / no access
    if not value:
        value = _read_env_file(name)    # fallback: uploaded secrets.env
    if value:
        os.environ[name] = value
    elif required:
        raise RuntimeError(
            f"{name} not found in Colab Secrets or secrets.env. "
            "Upload secrets.env to the Colab server and re-run."
        )
    return value

groq_key = get_secret("GROQ_API_KEY")


print("GROQ key loaded:", bool(groq_key))


GROQ key loaded: True


In [4]:
llm = ChatGroq(
        model = "openai/gpt-oss-120b",
        model_kwargs= {"seed" : 350 },
        temperature= 0,
        max_tokens= 250
)

**String output Parser**

In [5]:
template_h = ('human', 'Hey, who is Nvidia CEO ?')

response = llm.invoke([template_h])

response

AIMessage(content='Nvidia’s chief executive officer is **Jensen\u202fHuang**. He co‑founded the company in 1993 and has served as its CEO ever since.', additional_kwargs={'reasoning_content': 'The user asks: "Hey, who is Nvidia CEO ?" This is a straightforward factual question. The answer: Jensen Huang, co-founder and CEO of Nvidia. Provide current info. As of 2026, Jensen Huang remains CEO. Provide brief info.'}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 78, 'total_tokens': 174, 'completion_time': 0.202985683, 'completion_tokens_details': {'reasoning_tokens': 52}, 'prompt_time': 0.003013507, 'prompt_tokens_details': None, 'queue_time': 0.230086037, 'total_time': 0.20599919}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_86dc843b12', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1cad4-6957-754d-8cb7-f1f25bec829f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata

In [ ]:
string_parser = StrOutputParser()

parsed_data = string_parser.invoke(response)

parsed_data

'Nvidia’s chief executive officer is **Jensen\u202fHuang**. He co‑founded the company in 1993 and has served as its CEO ever since.'

**List Output Parser**

In [7]:
list_sep = CommaSeparatedListOutputParser().get_format_instructions()

print(list_sep)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [8]:
template_h = (
    'human',  
    f'''can you list down some exciting projects of nvidia ?  
       {list_sep}
    '''
    )

response = llm.invoke([template_h])

response

AIMessage(content='CUDA, RTX, DGX, Omniverse, Clara, Jetson, GeForce\u202fNOW, NVIDIA\u202fDRIVE, NVIDIA\u202fAI\u202fEnterprise, NVIDIA\u202fNeMo, NVIDIA\u202fIsaac, NVIDIA\u202fBroadcast, NVIDIA\u202fMaxine, NVIDIA\u202fTensorRT, NVIDIA\u202fG-SYNC, NVIDIA\u202fShield, NVIDIA\u202fBlueField, NVIDIA\u202fGrace, NVIDIA\u202fH100, NVIDIA\u202fRiva, NVIDIA\u202fModulus, NVIDIA\u202fNeural\u202fGraphics, NVIDIA\u202fCanvas, NVIDIA\u202fAudio2Face, NVIDIA\u202fRAPIDS,', additional_kwargs={'reasoning_content': 'The user asks: "can you list down some exciting projects of nvidia ? Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`". So we need to output a list of Nvidia projects, separated by commas. Should we include spaces? Both examples show both with spaces and without. Probably okay either way. Provide a list like: "CUDA, RTX, DGX, Omniverse, Clara, Jetson, GeForce NOW, Deep Learning AI, NVIDIA DRIVE, NVIDIA AI Enterprise". Ensure it\'s a singl

In [9]:
Parsed_list = CommaSeparatedListOutputParser()

parsed_list_response = Parsed_list.invoke(response)

parsed_list_response

['CUDA',
 'RTX',
 'DGX',
 'Omniverse',
 'Clara',
 'Jetson',
 'GeForce\u202fNOW',
 'NVIDIA\u202fDRIVE',
 'NVIDIA\u202fAI\u202fEnterprise',
 'NVIDIA\u202fNeMo',
 'NVIDIA\u202fIsaac',
 'NVIDIA\u202fBroadcast',
 'NVIDIA\u202fMaxine',
 'NVIDIA\u202fTensorRT',
 'NVIDIA\u202fG-SYNC',
 'NVIDIA\u202fShield',
 'NVIDIA\u202fBlueField',
 'NVIDIA\u202fGrace',
 'NVIDIA\u202fH100',
 'NVIDIA\u202fRiva',
 'NVIDIA\u202fModulus',
 'NVIDIA\u202fNeural\u202fGraphics',
 'NVIDIA\u202fCanvas',
 'NVIDIA\u202fAudio2Face',
 'NVIDIA\u202fRAPIDS',
 '']

**Date time Parser**

In [30]:
DatetimeOutputParser().get_format_instructions()

"Write a datetime string that matches the following pattern: '%Y-%m-%dT%H:%M:%S.%fZ'.\n\nExamples: 2023-07-04T14:30:00.000000Z, 1999-12-31T23:59:59.999999Z, 2025-01-01T00:00:00.000000Z\n\nReturn ONLY this string, no other words!"

In [29]:
template_h = ('human', f'''
give me the date of launch of Voyager 1

{DatetimeOutputParser().get_format_instructions()}
''')

response = llm.invoke([template_h])

#print(response)

date_parser = DatetimeOutputParser()

parsed_response = date_parser.invoke(response)

print(parsed_response)

1977-09-05 12:56:00
